# 六足ロボットの円柱押し：学生用APIを一歩ずつ学ぶ

**報酬・観測・行動のどこを変えると、学習結果がどう変わるか**を調べる教材です。
18関節の六足ロボット、固定した歩行モデル、MuJoCoの接触物理を使い、上位の動作をTorchRL PPOで学習します。

1. [Colabでデモを開く](https://colab.research.google.com/github/hashimoto-robotics-lab/hexapod-transport-rl/blob/main/notebooks/student_api_colab.ipynb)をクリックします。
2. 「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選びます。CPUでも短いデモは動きます。
3. 最初のセルでGitHubから教材をcloneし、依存ライブラリを準備します。
4. 上から順番に実行します。初回のインストール・GPUカーネルのコンパイルには数分かかる場合があります。

標準課題：円柱の周囲でロボットの初期位置・向きがランダム、目標方向・距離もランダム。
成功は**位置誤差5 cm未満・円柱速度0.06 m/s未満を0.4秒保持**です。円柱の向きは採点しません。
このノートブックは初期設定では短い**操作練習**です。本学習への切り替えはStep 1で行います。

## Step 0 — GitHubから教材を取得する

最初の実行でGitHubからcloneします。取得先は`REPOSITORY`、ブランチは`REVISION`で指定できます。
既定は公開リポジトリの`main`です。API・機体・歩行モデル・参照方策をまとめて取得します。
再実行時は取得済み教材を使い、編集したファイルを保持します。
最新版を取り直す場合は、新しいColabセッションで実行してください。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = os.environ.get(
    "HEXAPOD_REPOSITORY", "https://github.com/hashimoto-robotics-lab/hexapod-transport-rl.git"
)
REVISION = os.environ.get("HEXAPOD_REVISION", "main")  # 取得する公開済みブランチ
PROJECT_DIR = Path(os.environ.get("HEXAPOD_PROJECT_DIR", "/content/hexapod_transport_rl")).resolve()

if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REVISION,
        REPOSITORY, str(PROJECT_DIR),
    ], check=True)

required = ("src/hexapod_transport_rl/student.py", "tools/colab_setup.py",
            "checkpoints/cylinder_push.pt")
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError(
        f"教材ファイルがありません: {missing}。取得先のブランチREVISIONを確認してください。"
        "取得済みの教材が古い場合は新しいセッションで再実行してください。"
    )

sys.path.insert(0, str(PROJECT_DIR / "tools"))
from colab_setup import prepare_colab

prepare_colab(PROJECT_DIR, install=os.environ.get("HEXAPOD_SKIP_INSTALL") != "1")


## Step 1 — 実験の長さと保存先を決める

`PROFILE = "demo"`は、1エピソード2秒・小さなPPO更新を2回行う操作練習です。
この短さでは円柱を目標まで運ぶ能力は評価できません。成功率0でも、APIの接続確認としては正常です。

`"research"`は1エピソード40秒・655,360サンプルで学習する設定例です。
GPUで使ってください。比較は各条件3つの学習seed・共通の未学習評価seedで行います。
計算時間と到達する成功率はGPU・設定によって変わります。特にランダムな重みからの学習には追加の予算が必要になる場合があります。
保存先は実行ごとに新しく作ります。

In [ ]:
import json
import time
from dataclasses import replace
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Video, display

from hexapod_transport_rl import (
    ContinuousVelocity, CylinderPushTask, DiscreteDirections, ObservationTerm,
    PPOSettings, compare, evaluate, load_task, load_task_file, record_video, train,
)

PROFILE = "demo"  # 本学習へ進むときは "research" に変更して、このセル以降を再実行
BACKEND = "warp" if torch.cuda.is_available() else "cpu"
if PROFILE not in ("demo", "research"):
    raise ValueError('PROFILE は "demo" または "research" です')
if PROFILE == "research" and BACKEND != "warp":
    raise RuntimeError("本学習にはGPUランタイムを選択してください")

base = CylinderPushTask()  # 回転対称化なし。報酬・観測・行動を公平に変更できる共通条件
if PROFILE == "demo":
    base = replace(base, config=replace(base.config, episode_seconds=2.0))
    worlds, horizon = (8 if BACKEND == "warp" else 2), 8
    options = dict(steps=worlds * horizon * 2, num_envs=worlds, horizon=horizon,
                   backend=BACKEND, validation_envs=4 if BACKEND == "warp" else 2,
                   validate_every=1, settings=PPOSettings(epochs=1, minibatch_size=64))
    evaluation_episodes, comparison_seeds = 4, (42, 43)
else:
    options = dict(steps=655_360, num_envs=256, horizon=64, backend=BACKEND,
                   validation_envs=256, validate_every=10, settings=PPOSettings())
    evaluation_episodes, comparison_seeds = 256, (42, 43, 44)

WORK_DIR = PROJECT_DIR / "runs" / ("tutorial_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
WORK_DIR.mkdir(parents=True)
print("モード:", PROFILE, "物理計算:", BACKEND, "保存先:", WORK_DIR)
print("1エピソード:", base.config.episode_seconds, "秒 / 各学習:", options["steps"], "サンプル")

## Step 2 — `reset`・`step`・観測・報酬を見てみる

環境は`task.make_env()`で作り、`reset()`で開始、`step()`で0.2秒進めます。
`num_envs`は独立した世界の数で、各世界に六足ロボットは1台ずつ存在します。
`observation`が方策に渡る入力です。標準は14成分で、行動は停止＋世界座標の8方向の9択です。
`env.state`では、報酬を設計するときに使える名前付き物理量を確認できます。

In [ ]:
env = base.make_env(num_envs=1, backend="cpu", render_mode="rgb_array")
try:
    env.set_seed(42)
    td = env.reset()
    print("方策の観測:", td["observation"].shape, td["observation"][0].tolist())
    print("観測の内訳:", env.observation_layout)
    print("ロボット位置 [m]:", env.state.robot_position[0].tolist())
    print("円柱位置 [m]:", env.state.cargo_position[0].tolist())
    print("目標位置 [m]:", env.state.goal_position[0].tolist())
    initial_image = env.worlds.envs[0].render()
    td["action"] = torch.tensor([1])  # 0=停止、1=東、2=北東、3=北、…、8=南東
    after = env.step(td)["next"]
    print("1ステップの報酬:", after["reward"].item())
    print("内訳:", {name: value.item() for name, value in after["reward_terms"].items()})
    print("残りの目標距離 [m]:", after["distance"].item())
finally:
    env.close()

plt.figure(figsize=(6, 4))
plt.imshow(initial_image)
plt.axis("off")
plt.show()

## Step 3 — 標準設定でPPOを動かす

`train(task, 保存先, ...)`が、収集・PPO更新・検証・保存をまとめて実行します。
ここでは**ランダムな重みから**学習します。固定歩行モデルは再学習しません。
`latest.pt`は最後の更新、`best.pt`は共通の検証条件で選んだ重みです。
学習前の重みが検証で優れていた場合には、それが`best.pt`になることもあります。

In [ ]:
started = time.perf_counter()
run = train(base, WORK_DIR / "baseline", seed=42, **options)
print(f"学習・検証・保存の実時間: {time.perf_counter() - started:.1f} 秒")
print("選択した重み:", run.checkpoint)
print("最後の重み:", run.latest)

history = pd.read_json(run.directory / "progress.json")
display(history[["steps", "reward", "mean_distance_m", "successes"]])

## Step 4 — 未学習seedで評価し、動画を残す

`evaluate`は学習に使っていないseedで初期状態を作り、成功数・距離を測ります。
`video`は実際のMuJoCoの動きを25 fpsで保存します。動画1本の成功・失敗と、複数回評価した成功率を区別してください。
`demo`では2秒で時間切れになります。40秒の参照方策はStep 9で確認します。

In [ ]:
report = run.evaluate(episodes=evaluation_episodes, seed=240000, backend=BACKEND)
print(f"成功: {report['successes']} / {report['episode_count']}")
print(f"平均の最終目標距離: {report['mean_distance_m']:.3f} m")
video_result = run.video(WORK_DIR / "baseline.mp4", seed=240000)
print(video_result)
display(Video(video_result["video"], embed=True, width=640))

## Step 5 — 報酬の係数を1つ変える

「円柱を目標へ近づけたこと」への報酬`goal_progress`を20から40へ変更します。
`replace`は元の課題を残して、新しい課題を作ります。物理・観測・行動・成功条件は共通です。
報酬の数値が大きくなるだけで学習が良くなるとは限りません。最終的な成功率と距離で比較します。

In [ ]:
reward_task = replace(base, rewards={**base.rewards, "goal_progress": 40.0})
display(pd.DataFrame({
    "標準の重み": {name: term.weight for name, term in base.rewards.items()},
    "変更後の重み": {name: term.weight for name, term in reward_task.rewards.items()},
}))

## Step 6 — 観測を減らす・行動を変える

速度の情報を外し、円柱からロボット・目標へのXYとロボットの向きだけを入力します（6成分）。
`scale`で物理量を割ります。XYは円柱の直径0.5 mで割り、標準観測と同じ倍率を使います。
行動は`DiscreteDirections(4)`なら停止＋東西南北の5択、`ContinuousVelocity()`なら連続XYの2成分です。
行動方向は初期設定では世界座標です。`frame="body"`で機体座標へ切り替えられます。

入力や出力の意味が変わるときは、既存の重みを引き継がず新規に学習します。

In [ ]:
observation_task = replace(base, observations=(
    ObservationTerm("cargo_to_robot", scale=base.config.width),
    ObservationTerm("cargo_to_goal", scale=base.config.width),
    "robot_heading",
))
action_task = replace(base, actions=DiscreteDirections(count=4, include_stop=True))
continuous_task = replace(base, actions=ContinuousVelocity(frame="world", speed=0.15))

for name, task in {"6成分の観測": observation_task,
                   "停止＋4方向": action_task, "連続XY": continuous_task}.items():
    probe = task.make_env(num_envs=1, backend="cpu")
    try:
        print(name, "/ 観測:", tuple(probe.reset()["observation"].shape),
              "/ 行動:", probe.action_spec)
    finally:
        probe.close()

## Step 7 — 独自の報酬・観測をPython関数で追加する

円柱が目標10 cm以内に来たときの速度を抑える報酬と、円柱表面までの距離の観測を追加します。
関数は`PushState`の名前付き物理量からTensorを計算するだけです。
各世界1つの報酬は`Tensor[B]`、観測は`Tensor[B]`または`Tensor[B, D]`を返します。
`per_second=True`の継続コストにはAPIが`dt`を掛けます。

独自関数は**小さな`.py`ファイル**に書きます。APIがそのソースを重みと一緒に保存するため、後から同じ定義で評価できます。
ノートブック上のセルだけで定義した関数やlambdaは登録できません。次のセルの関数本体を編集して試してください。

In [ ]:
%%writefile {WORK_DIR}/my_task.py
from hexapod_transport_rl import CylinderPushTask, ObservationTerm, RewardTerm

def surface_distance(state):
    return state.surface_gap

def near_goal_speed(before, after):
    return after.cargo_speed.square() * (after.goal_distance < 0.10)

original = CylinderPushTask()
TASK = CylinderPushTask(
    observations=(*original.observations,
                  ObservationTerm(surface_distance, scale=0.5)),
    rewards={**original.rewards,
             "near_goal_speed": RewardTerm(near_goal_speed, weight=-2.0, per_second=True)},
)

In [ ]:
custom_task = load_task_file(WORK_DIR / "my_task.py")
custom_task = replace(custom_task, config=base.config)
custom_run = train(custom_task, WORK_DIR / "custom", seed=42, **options)
custom_report = custom_run.evaluate(episodes=evaluation_episodes, seed=240000, backend=BACKEND)
print("観測成分:", custom_report["policy_input_dim"], "/ 成功数:", custom_report["successes"])
print("保存した独自関数:", list((custom_run.directory / "task_sources").glob("*.py")))
restored = load_task(custom_run.checkpoint)
assert restored.policy_signature() == custom_task.policy_signature()
print("保存した観測・報酬・行動の定義を再読み込みできました。")

## Step 8 — 同じ条件で複数seedの比較をする

標準・報酬の変更・観測の変更・行動の変更を、共通のサンプル数と評価seedで比較します。
**各条件で1か所ずつ変える**と、結果を解釈しやすくなります。
学習seedはPPOの初期重みと収集状態を変えます。評価seedは全条件で共通です。
`demo`の小さな成功率は操作練習の結果です。卒論では`research`に切り替え、十分な学習予算と評価回数を使ってください。
このセルは4条件×seed数だけ新規学習を実行します。

In [ ]:
tasks = {"baseline": base, "reward": reward_task,
         "observation": observation_task, "action": action_task}
comparison = compare(
    tasks, WORK_DIR / "comparison", seeds=comparison_seeds,
    evaluation_episodes=evaluation_episodes, evaluation_seed=240000, **options,
)
results = pd.read_csv(WORK_DIR / "comparison/comparison.csv")
display(results)
display(pd.DataFrame(comparison["summary"]).T)
print("保存先:", WORK_DIR / "comparison/comparison.csv")

## Step 9 — 既存の学習済み方策を再利用する

同梱の`cylinder_push.pt`は、これまでの実験で学習した参照方策です。
`load_task`でその重みと一致する観測・行動・前処理を復元し、40秒の標準課題を評価します。
この方策は**目標方向に合わせて45度刻みで座標を回す前処理**を使っています。
前処理を使わないStep 3の新規学習とは別条件なので、そのまま学習性能を比較しないでください。

続きから学習するときは、`train(..., initial_checkpoint=reference)`を指定できます。
互換性のある観測・行動のまま報酬を変更する用途で使います。

In [ ]:
reference = PROJECT_DIR / "checkpoints/cylinder_push.pt"
reference_task = load_task(reference)
print("既存方策の前処理:", reference_task.rotation)
print("評価の制限時間:", reference_task.config.episode_seconds, "秒")
reference_report = evaluate(reference, episodes=evaluation_episodes, seed=250000, backend=BACKEND)
print(f"参照方策の成功: {reference_report['successes']} / {reference_report['episode_count']}")
reference_video = record_video(reference, WORK_DIR / "reference.mp4", seed=250000)
display(Video(reference_video["video"], embed=True, width=640))

# 続きから学習する場合（任意。新しい出力先を指定）:
# continued = train(reference_task, WORK_DIR / "continued", seed=46,
#                   initial_checkpoint=reference, **options)

## Step 10 — 結果を保存して、次の実験へ

`task.json`に設定、`task_sources/`に独自関数、`environment.json`に依存ライブラリの版が残ります。
`progress.json`は学習経過、`validations.json`は検証、`evaluation_*.json`は独立評価です。
Colabを終了する前に結果をダウンロードしてください。学習済み重みだけでなく、フォルダごと保存すると独自関数も持ち出せます。

次の課題例：距離の重みだけ変える／速度の観測を1つずつ外す／停止の有無や4・8・16方向を比較する。
仮説・変更した項目・seed・学習サンプル数・成功率・最終距離を記録すると、卒論の実験表につなげられます。

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK_DIR), "zip", root_dir=WORK_DIR.parent, base_dir=WORK_DIR.name)
print("実験結果ZIP:", archive)

DOWNLOAD_RESULTS = False  # Colabで手元に保存するときは True に変更
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)

詳しい項目一覧は教材内の`docs/student-api.md`、通常のPython実験は`examples/student/`を参照してください。

環境の基準：Python 3.12/3.13、PyTorch 2.11.0、TorchRL 0.14.0、TensorDict 0.14.2、MuJoCo 3.11.0。
GPUではCUDA 12.8版PyTorchとMuJoCo Warp 3.11.0 / Warp 1.15.0を準備します。
Colabでライブラリがすでに読み込まれていた場合は、準備セルの案内に従ってセッションを再起動してください。
Colab標準環境は更新されるため、教材の準備セルで依存ライブラリを指定しています。
[Colabのランタイム](https://research.google.com/colaboratory/runtime-version-faq.html) /
[PyTorchの指定版インストール](https://pytorch.org/get-started/previous-versions/)